# single-cell foundation model 기초: cell embedding과 annotation

single-cell RNA 발현을 foundation model 입력으로 바꿔 cell embedding, annotation, perturbation으로 해석하고, 실제 Geneformer로 PBMC 세포를 임베딩한 결과를 어디까지 주장할 수 있는지 점검합니다.

## 강의 목표

- cell × gene matrix, QC, normalization, HVG가 모델 입력과 어떻게 연결되는지.
- Geneformer의 rank-value와 scGPT의 gene+expression이 무엇을 다르게 보존하는지.
- 실제 Geneformer로 PBMC3k 임베딩을 추출해 PCA baseline과 비교.
- annotation, probing, batch integration, perturbation, target prioritization.
- UMAP 모양이나 embedding 거리로 과대해석하지 않는 claim level.

## 참고 자료

- Scanpy: Wolf et al., *Genome Biology* 2018. DOI: https://doi.org/10.1186/s13059-017-1382-0
- UMAP: McInnes et al., *JOSS* 2018. DOI: https://doi.org/10.21105/joss.00861
- Geneformer: Theodoris et al., *Nature* 2023. DOI: https://doi.org/10.1038/s41586-023-06139-9
- scGPT: Cui et al., *Nature Methods* 2024. DOI: https://doi.org/10.1038/s41592-024-02201-0
- scVI: Lopez et al., *Nature Methods* 2018. DOI: https://doi.org/10.1038/s41592-018-0229-2
- PBMC 3k public dataset: 10x Genomics / `scanpy.datasets.pbmc3k()`
- Geneformer model family: https://huggingface.co/ctheodoris/Geneformer

## 실습 범위

기본 경로는 PBMC3k(약 2,700개 말초혈액 단핵세포)를 Geneformer V1-10M으로 임베딩합니다. Colab에서 런타임 유형을 T4 GPU로 바꿔 실행하고, 다운로드나 GPU가 막히면 synthetic 데이터와 PCA 임베딩이 대신 동작합니다.

**주의:** 여기서 만드는 cell type은 marker rule의 가설적 주석이며 정답 라벨이 아닙니다. 어떤 결과도 disease biology나 therapeutic target 결론으로 바로 확장하지 않습니다.

## 실행 스위치: 실제 모델 경로와 fallback

맨 위 flag가 경로를 정합니다. 기본 경로는 Geneformer V1-10M으로 raw count를 rank-value 토큰화해 256차원 임베딩을 뽑고(GPU 전제), fallback은 다운로드가 막히면 synthetic 데이터의 PCA 임베딩을 씁니다. `USE_GENEFORMER`(기본 `True`)와 `N_CELLS_EMBED`(600)가 주요 스위치입니다.

**핵심:** 하나의 flag가 Geneformer 경로와 synthetic fallback을 가릅니다.

In [ ]:
USE_GENEFORMER = True

MODEL_REPO = "ctheodoris/Geneformer"
MODEL_SUBFOLDER = "Geneformer-V1-10M"
N_CELLS_EMBED = 600
N_NEIGHBORS = 12
RANDOM_STATE = 0

print({
    "USE_GENEFORMER": USE_GENEFORMER,
    "MODEL": f"{MODEL_REPO}/{MODEL_SUBFOLDER}",
    "N_CELLS_EMBED": N_CELLS_EMBED,
})


`USE_GENEFORMER`가 `True`이므로 실제 Geneformer를 돌립니다. `N_CELLS_EMBED`=600은 강의용 제한이며 실제 분석에서는 전체 세포를 씁니다.

**주의:** flag를 바꿨는데 이전 값이 그대로면 이 셀을 다시 실행하지 않은 것입니다.

## 실습 환경 준비

single-cell 표준 도구를 불러옵니다. `scanpy`(`sc`)는 AnnData로 데이터를 다루고 `anndata`(`ad`)는 그 구조입니다. import 실패 시 try/except가 `pip install` 후 다시 불러오므로 셀을 통째로 실행합니다.

In [ ]:
import subprocess, sys
try:
    import scanpy as sc
    import anndata as ad
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scanpy"])
    import scanpy as sc
    import anndata as ad
# Geneformer 경로에 필요한 패키지 (대개 Colab에 사전 설치됨) — 없을 때만 설치
for _pkg in ["huggingface_hub", "transformers"]:
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])
print("scanpy", sc.__version__)


수치 계산·시각화·sparse 도구입니다. `numpy`(`np`)와 `pandas`(`pd`)는 배열과 표, `matplotlib.pyplot`(`plt`)은 그림, `scipy.sparse`(`sp`)는 대부분 0인 count matrix, `display`는 표 출력을 담당합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from IPython.display import display


scikit-learn 도구입니다. `PCA`(차원 축소), `train_test_split`(reference/query 분할), `LogisticRegression`(probing), `silhouette_score`(cell type·batch 분리 정도)를 씁니다.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score


전역 설정을 잡습니다. 경고를 숨기고 scanpy 출력을 줄이며, `RANDOM_STATE`로 `rng`를 만들어 재현성을 확보합니다.

**주의:** 오류가 나면 대개 설치 문제이니 Colab이라면 런타임을 재시작하고 위에서부터 다시 실행합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams["figure.dpi"] = 120

def plot_2d(xy, labels, title):
    fig, ax = plt.subplots(figsize=(6.8, 5.2))
    for lab in pd.unique(labels):
        m = np.asarray(labels) == lab
        ax.scatter(xy[m, 0], xy[m, 1], s=10, alpha=0.7, label=str(lab))
    ax.set_title(title); ax.set_xlabel("axis 1"); ax.set_ylabel("axis 2")
    ax.legend(frameon=False, fontsize=8, markerscale=1.5)
    plt.tight_layout(); plt.show()

print("기본 패키지 import 완료")


## GPU device 잡기

`get_device`는 GPU가 있으면 `cuda`, 없으면 `cpu`를 돌려줍니다. CPU면 Geneformer forward가 느려지므로 Colab에서 런타임을 T4 GPU로 바꿉니다. 이 `device`는 모델과 입력을 같은 장치로 올릴 때 재사용합니다.

In [ ]:
import torch

def get_device():
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"GPU: {p.name} ({p.total_memory/1e9:.1f} GB) | torch {torch.__version__}, CUDA {torch.version.cuda}")
        return torch.device("cuda")
    print("⚠ GPU 미감지 — CPU(느림). Colab: 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 재실행.")
    return torch.device("cpu")

device = get_device()


## 1. cell × gene matrix: PBMC3k 불러오기

single-cell foundation model의 입력은 cell × gene count matrix입니다. 행은 cell, 열은 gene, 값은 count이고 cell type 같은 metadata가 붙습니다. 먼저 PBMC marker gene 묶음을 정의합니다(annotation·perturbation 대상 선정 기준).

**핵심:** 입력은 cell × gene count matrix와 그에 붙은 metadata입니다.

In [ ]:
MARKER_SETS = {
    "T cell": ["CD3D", "CD3E", "IL7R", "TRAC"],
    "B cell": ["MS4A1", "CD79A", "CD79B", "BANK1"],
    "NK cell": ["NKG7", "GNLY", "KLRD1", "PRF1"],
    "Monocyte": ["LYZ", "S100A8", "S100A9", "LST1"],
    "Dendritic": ["FCER1A", "CD1C", "CLEC10A"],
    "Megakaryocyte": ["PPBP", "PF4", "ITGA2B"],
}


다운로드가 막힐 때를 대비해 `make_example_pbmc_like`를 정의합니다. 각 세포에 cell type을 배정하고 marker 발현을 높인 뒤 noise를 더해 `AnnData`로 돌려줍니다. 실제 PBMC3k가 받아지면 쓰이지 않습니다.

In [ ]:
def make_example_pbmc_like(n_cells=720):
    genes = sorted({g for gs in MARKER_SETS.values() for g in gs}) + [f"NOISE{i:03d}" for i in range(90)]
    labels = rng.choice(list(MARKER_SETS), size=n_cells)
    gi = {g: i for i, g in enumerate(genes)}
    X = rng.normal(0.15, 0.08, size=(n_cells, len(genes)))
    for i, lab in enumerate(labels):
        for g in MARKER_SETS[lab]:
            X[i, gi[g]] += rng.normal(3.0, 0.4)
    X = np.clip(X, 0, None).astype("float32")
    obs = pd.DataFrame({"cell_type": labels}, index=[f"cell_{i:04d}" for i in range(n_cells)])
    return ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=pd.Index(genes, name="gene")))


`sc.datasets.pbmc3k()`는 약 2,700개 말초혈액 단핵세포 raw count matrix를 돌려줍니다. 실패하면 synthetic 데이터로 내려가고 `is_real` flag로 경로를 기록합니다.

**핵심:** Geneformer의 입력은 raw count이므로 이 단계의 raw count를 보존합니다.

In [ ]:
try:
    adata = sc.datasets.pbmc3k()
    adata.var_names_make_unique()
    is_real = True
    print("실제 PBMC3k 로드:", adata.shape)
except Exception as exc:
    print("PBMC3k 다운로드 실패 → synthetic PBMC-like 사용:", type(exc).__name__)
    adata = make_example_pbmc_like()
    is_real = False


`AnnData`는 표준 컨테이너입니다. `X`는 cell × gene 행렬, `obs`는 cell metadata(cell type 등), `var`는 gene metadata, `obsm`은 embedding 행렬입니다. 실제 분석에서는 gene symbol과 Ensembl ID, normalization 상태를 점검합니다.

In [ ]:
print(adata)
print("\nX dtype:", adata.X.dtype, "| sparse:", sp.issparse(adata.X))
display(adata.var.head())


**출력 읽기**

- `n_obs × n_vars`가 세포·gene 수이고, 실제 PBMC3k라면 약 2,700 × 32,738입니다.
- var index가 gene symbol(CD3D, MS4A1 등)인지 확인합니다.

Geneformer는 symbol을 Ensembl ID로 바꿔 token화하므로, symbol이 아니면 많은 gene이 빠집니다.

**핵심:** var index가 gene symbol이어야 Geneformer 토큰화에서 gene이 유지됩니다.

## 2. QC와 전처리: baseline을 위한 정규화

Geneformer는 raw count를 쓰지만 PCA baseline과 marker 해석에는 전처리가 필요합니다. raw count를 `layers["counts"]`에 복사해 두고 PCA용 사본만 정규화하며, QC 필터(pct_counts_mt < 5%, n_genes < 2500)로 죽어가는 세포와 doublet 의심 세포를 거릅니다(정식 검출은 Scrublet/SOLO).

**핵심:** raw count를 `layers["counts"]`에 보존하고, PCA용 사본에만 정규화를 적용합니다.

In [ ]:
if is_real:
    adata.layers["counts"] = adata.X.copy()
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    adata.var["mt"]   = adata.var_names.str.upper().str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.str.upper().str.startswith(("RPS", "RPL"))
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], percent_top=None, log1p=False, inplace=True)
    n0 = adata.n_obs
    adata = adata[(adata.obs["pct_counts_mt"] < 5) & (adata.obs["n_genes_by_counts"] < 2500)].copy()
    print(f"QC: min_genes/min_cells + mito<5% + n_genes<2500 -> {n0} -> {adata.n_obs} cells "
          f"| median mito%={adata.obs['pct_counts_mt'].median():.1f}")
adata_pp = adata.copy()
print("raw counts 보존:", "counts" in adata.layers)


실제 데이터일 때만 정규화합니다. `normalize_total`로 library size를 1e4로 맞춘 뒤 `log1p`로 로그 변환합니다. synthetic fallback은 이미 정리된 값이라 건너뜁니다.

In [ ]:
if is_real:
    sc.pp.normalize_total(adata_pp, target_sum=1e4)
    sc.pp.log1p(adata_pp)
    print("정규화 후:", adata_pp.shape)
else:
    print("synthetic fallback: 정규화를 생략하고 기존 값을 사용합니다.")


highly variable gene(HVG)은 세포 간 변이가 큰 gene으로 cell state 신호를 담습니다. `highly_variable_genes`로 상위 2,000개를 골라 `scale`로 평균 0, 분산 1로 맞춥니다.

**주의:** batch가 있는 실제 데이터에서는 `highly_variable_genes(..., batch_key="batch")`로 batch 편향 gene을 억제합니다(이 데이터에는 batch가 없어 생략).

In [ ]:
if is_real:
    sc.pp.highly_variable_genes(adata_pp, n_top_genes=2000, flavor="seurat")
    adata_hvg = adata_pp[:, adata_pp.var["highly_variable"]].copy()
    sc.pp.scale(adata_hvg, max_value=10)
else:
    adata_hvg = adata_pp.copy()
print("HVG matrix:", adata_hvg.shape)


PCA로 차원을 줄이고 neighbors와 UMAP을 계산합니다. PCA는 HVG 발현을 소수 축으로 압축한 baseline 표현이고, 이후 Geneformer 임베딩과 비교할 기준이 됩니다.

**주의:** UMAP은 시각화 도구이지 cell type annotation의 정답지가 아닙니다.

In [ ]:
sc.pp.pca(adata_hvg, n_comps=30, svd_solver="arpack", random_state=RANDOM_STATE)
sc.pp.neighbors(adata_hvg, n_neighbors=N_NEIGHBORS, n_pcs=30, random_state=RANDOM_STATE)
sc.tl.umap(adata_hvg, random_state=RANDOM_STATE)
print("PCA/neighbors/UMAP 완료 | X_pca:", adata_hvg.obsm["X_pca"].shape)


## 3. cell을 token으로: rank-value와 gene+expression

DNA 서열과 달리 cell × gene matrix의 열 순서에는 고정된 의미가 없습니다. 그래서 모델마다 cell을 token으로 바꾸는 방식이 다르고, 그것이 모델의 생물학적 가정을 정합니다. Geneformer는 발현이 높은 gene부터 순위를 매기는 rank-value 방식으로 상대적 우선순위를 강조합니다.

**핵심:** cell × gene matrix에는 자연스러운 순서가 없으므로 tokenizer 설계가 모델의 가정을 정합니다.

In [ ]:
def counts_matrix():
    return adata.layers["counts"] if "counts" in adata.layers else adata.X

def dense_row(M, i):
    r = M[i]
    return np.asarray(r.todense()).ravel() if sp.issparse(r) else np.asarray(r).ravel()


한 세포의 gene 순위를 두 방식으로 비교합니다. 왼쪽 raw count 순위는 B2M이나 ribosomal(RPS/RPL) 같은 광범위 발현 gene이 위에 옵니다. 오른쪽 정규화 순위(`count / gene별 corpus median`)는 이런 housekeeping이 밀려나 Geneformer가 보는 순서에 가깝습니다. 구현은 섹션 5의 토큰화 셀에 있습니다.

**핵심:** raw 상위 gene과 Geneformer 상위 gene은 다르며, 이 구분이 섹션 10 perturbation 해석의 핵심입니다.

In [ ]:
ci = 0
row0 = dense_row(counts_matrix(), ci).astype(float)
syms = np.asarray(adata.var_names)
Mc_all = counts_matrix()
present = np.where(row0 > 0)[0]


각 gene을 그 gene의 대표 발현(이 데이터에서 계산한 '0이 아닌 값의 median')으로 나눠 정규화값을 구합니다. 어디서나 높게 발현되는 housekeeping은 median이 커서 값이 눌리고 순위가 내려갑니다.

In [ ]:
# Geneformer식 정규화의 self-contained 근사: 각 gene을 "그 gene의 0이 아닌 값들의 median"으로 나눔
Xcsc = Mc_all.tocsc() if sp.issparse(Mc_all) else np.asarray(Mc_all)
def nz_median(j):
    d = Xcsc[:, int(j)].data if sp.issparse(Xcsc) else Xcsc[:, int(j)][Xcsc[:, int(j)] > 0]
    return float(np.median(d)) if len(d) else 1.0
med = {int(j): nz_median(j) for j in present}
norm_val = np.array([row0[j] / med[int(j)] for j in present])
demo = pd.DataFrame({"gene": syms[present], "count": row0[present], "norm": np.round(norm_val, 2)})
print(f"cell: {adata.obs_names[ci]} | 검출 gene {len(present)}개 | 정규화값 계산 완료")

왼쪽은 raw count 순위, 오른쪽은 정규화 순위입니다. 정규화 쪽에서 housekeeping(B2M, RPS 등)이 밀려나고 그 세포에 상대적으로 특이한 gene이 위로 올라옵니다. 오른쪽이 Geneformer가 실제로 보는 순서에 가깝습니다.

**핵심:** 이 정규화 순위가 섹션 5의 토큰화 셀에서 실제 Geneformer 입력이 됩니다.

In [ ]:
raw_rank  = demo.nlargest(10, "count")[["gene", "count"]].reset_index(drop=True)
norm_rank = demo.nlargest(10, "norm")[["gene", "norm"]].reset_index(drop=True)
ranked = pd.concat([raw_rank.add_prefix("raw_"), norm_rank.add_prefix("normalized_")], axis=1)
print("cell:", adata.obs_names[ci])
print("좌: raw count 순위(housekeeping/ribosomal 상위) | 우: gene별 정규화 순위 ≈ Geneformer가 보는 순서(정확한 구현은 섹션 5의 토큰화 셀)")
display(ranked)


scGPT 계열은 순위만 쓰는 대신 gene identity와 expression value를 함께 token으로 넣고 batch·condition token도 더합니다. 표로 펼치면 gene token, expression value, condition이 한 줄에 묶입니다.

**핵심:** 같은 데이터라도 tokenizer에 따라 모델이 보는 입력이 달라집니다.

In [ ]:
scgpt_like = pd.DataFrame({"gene": np.asarray(adata.var_names), "count": row0}).nlargest(10, "count").reset_index(drop=True)
scgpt_like["gene_token"] = scgpt_like["gene"]
scgpt_like["expression_value"] = scgpt_like["count"]
scgpt_like["condition_batch"] = "cond_example"   # 조건/batch token 예시(값 자체는 illustrative)
display(scgpt_like[["gene_token", "expression_value", "condition_batch"]].head(10))


입력 방식을 한 표로 비교합니다. 같은 cell vector라도 binned, rank-ordered, gene+value, gene module 중 무엇으로 바꾸느냐에 따라 보존 정보가 달라집니다.

**핵심:** 모델을 고르기 전에 그 모델의 tokenization 방식을 먼저 확인합니다.

In [ ]:
tok_compare = pd.DataFrame([
    {"방식": "binned expression", "대표 모델": "scBERT", "가정": "expression 값을 몇 단계 token으로 나눔"},
    {"방식": "rank-ordered genes", "대표 모델": "Geneformer", "가정": "cell 안 gene 순위가 cell state를 담음"},
    {"방식": "gene + expression", "대표 모델": "scGPT, scFoundation", "가정": "gene identity와 expression value를 함께 사용"},
    {"방식": "gene module/patch", "대표 모델": "CellPatch 등", "가정": "co-expression group으로 sequence length 축소"},
])
display(tok_compare)


## 4. Marker 기반 cell type annotation

PBMC3k에는 정답 라벨이 없어 marker gene 발현으로 pseudo-label을 만듭니다. cell type별 marker set 평균 발현을 계산해 가장 높은 type을 붙이며, marker rule의 산물이라 marker 누락이나 dropout에 흔들립니다.

**핵심:** 이 cell type 라벨은 marker rule에서 나온 pseudo-label이며 정답이 아닙니다.

In [ ]:
score_df = pd.DataFrame(index=adata_pp.obs_names)
for ct, markers in MARKER_SETS.items():
    present = [g for g in markers if g in adata_pp.var_names]
    if present:
        sub = adata_pp[:, present].X
        score_df[ct] = np.asarray(sub.mean(axis=1)).ravel() if sp.issparse(sub) else np.asarray(sub).mean(axis=1)
print("marker score(cell type별 marker 평균 발현) 계산:", list(score_df.columns))


세포마다 가장 높은 marker score의 cell type을 붙이고 세포 수를 출력합니다. PBMC에서는 보통 T cell이 많고 dendritic cell이나 megakaryocyte는 드뭅니다.

**주의:** 이 '평균 + 최고점 배정'은 가장 단순한 heuristic으로, 스케일 차이와 문턱 부재로 애매한 세포도 강제 배정됩니다. 실제로는 clustering(Leiden) 후 cluster 단위 주석이나 `sc.tl.score_genes`, CellTypist를 씁니다.

In [ ]:
if is_real and len(score_df.columns) > 0:
    adata.obs["cell_type"] = score_df.idxmax(axis=1).values
display(adata.obs["cell_type"].value_counts().rename_axis("cell_type").reset_index(name="n"))


이제 PCA/UMAP baseline 좌표 위에 annotation을 그립니다. 그림 helper `plot_2d`는 앞의 import/셋업 파트에서 미리 정의해 두었으므로, 셀을 순서와 무관하게 다시 실행해도 안전합니다. 먼저 UMAP을 cell type 색으로 칠해 baseline 표현이 면역세포 종류를 어느 정도 나누는지 봅니다.

In [ ]:
umap_xy = adata_hvg.obsm["X_umap"]
ct_labels = adata.obs["cell_type"].to_numpy() if "cell_type" in adata.obs else adata.obs.iloc[:, 0].to_numpy()
plot_2d(umap_xy, ct_labels, "PCA/UMAP baseline colored by cell type")


**주의:** UMAP은 visualization이지 annotation의 정답지가 아닙니다. 가깝다고 같은 cell type인 것도, 멀다고 다른 기능인 것도 아닙니다. baseline이 이미 cell type을 잘 나눈다면 Geneformer 임베딩이 무엇을 더하는지 더 엄격하게 물어야 합니다.

### cell embedding이란

cell embedding은 세포 상태를 요약한 고정 길이 벡터입니다(Geneformer는 256차원). 가까운 두 세포는 상태가 비슷해 거리·군집·분류가 의미를 가집니다.

- PCA는 이 데이터에서 학습하지만, Geneformer는 3천만 세포로 미리 학습합니다.
- PCA는 선형 축이지만, Geneformer는 attention으로 gene 사이 문맥을 반영합니다.

**핵심:** PCA는 이 데이터의 분산 방향, Geneformer embedding은 대규모 사전학습으로 만든 표현입니다.

## 5. 실제 Geneformer 세포 임베딩

Geneformer는 약 30M human single-cell transcriptome으로 학습해, rank-ordered gene sequence를 받아 256차원 임베딩을 만듭니다. cell type별로 균형 있게 `N_CELLS_EMBED`개를 추립니다.

**주의:** `N_CELLS_EMBED`(=600)는 상한입니다. 희귀 세포(Dendritic, Megakaryocyte)가 부족하면 실제 임베딩 세포 수는 600보다 작을 수 있습니다.

In [ ]:
ct_arr = adata.obs["cell_type"].to_numpy()
n_each = max(1, N_CELLS_EMBED // len(pd.unique(ct_arr)))
sub_idx = []
for ct in pd.unique(ct_arr):
    pos = np.where(ct_arr == ct)[0]
    sub_idx.extend(rng.choice(pos, size=min(len(pos), n_each), replace=False))
sub_idx = np.array(sorted(sub_idx))
sub_obs = adata.obs.iloc[sub_idx].reset_index(drop=True)
symbols = np.asarray(adata.var_names)
Mc = counts_matrix()
count_rows = [dense_row(Mc, i) for i in sub_idx]
print("임베딩 대상 세포:", len(sub_idx))


아래 셀이 기본 경로, 즉 실제 Geneformer로 임베딩을 뽑는 부분입니다. gene dictionary 3개와 Geneformer V1-10M을 내려받아 토큰화·forward pass·mean-pooling으로 세포당 256차원 벡터를 만들고, 실패하면 다음 셀 fallback으로 넘어갑니다.

**주의:** 다운로드와 forward pass로 수 분 걸릴 수 있고 진행 막대가 멈춘 듯 보여도 정상입니다. 로드 중 `pooler weights ... newly initialized` 경고는 pooler 대신 `last_hidden_state`를 직접 mean-pooling하므로 무시해도 됩니다. 출력의 `embedding source`가 'Geneformer ...'인지 'fallback ...'인지 확인합니다.

### 이 셀은 4단계를 한 셀에 합칩니다

1. 로드. gene dictionary 3개(token / median / name→id)와 Geneformer V1-10M을 내려받습니다.
2. 토큰화. raw count를 섹션 3의 정규화 순위(`count / gene median`)로 바꿔 상위 gene token id 시퀀스를 만듭니다.
3. forward pass. token 시퀀스로 `last_hidden_state`(token별 문맥 벡터)를 얻습니다.
4. mean-pooling. 그 벡터를 padding 제외 평균 내어 세포당 256차원 벡터로 만듭니다.

**핵심:** 네 단계로 세포당 256차원 임베딩을 만듭니다.

In [ ]:
cell_emb = None
embedding_source = None
if USE_GENEFORMER:
    try:
        import pickle
        from huggingface_hub import hf_hub_download
        from transformers import AutoModel
        def _load_pkl(fn):
            return pickle.load(open(hf_hub_download(MODEL_REPO, fn), "rb"))
        token_dict  = _load_pkl("geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl")
        median_dict = _load_pkl("geneformer/gene_dictionaries_30m/gene_median_dictionary_gc30M.pkl")
        name_id     = _load_pkl("geneformer/gene_dictionaries_30m/gene_name_id_dict_gc30M.pkl")
        gf_model = AutoModel.from_pretrained(MODEL_REPO, subfolder=MODEL_SUBFOLDER).to(device).eval()

        def tokenize_cell(counts, syms, max_len=2048):
            toks, vals = [], []
            for sym, v in zip(syms, counts):
                if v <= 0:
                    continue
                ens = name_id.get(sym)
                if ens is None or ens not in median_dict or ens not in token_dict:
                    continue
                toks.append(token_dict[ens]); vals.append(v / median_dict[ens])
            order = np.argsort(vals)[::-1][:max_len]
            return [toks[i] for i in order]

        def geneformer_embed(rows, syms, batch_size=16):
            pad = token_dict.get("<pad>", 0); out = []
            ids_all = [tokenize_cell(r, syms) or [pad] for r in rows]
            for s in range(0, len(ids_all), batch_size):
                chunk = ids_all[s:s+batch_size]; ml = max(len(x) for x in chunk)
                ids = torch.tensor([x + [pad]*(ml-len(x)) for x in chunk], device=device)
                attn = (ids != pad).long()
                with torch.no_grad():
                    h = gf_model(input_ids=ids, attention_mask=attn).last_hidden_state
                m = attn.unsqueeze(-1).to(h.dtype)
                out.append(((h*m).sum(1)/m.sum(1).clamp(min=1)).float().cpu().numpy())
            return np.concatenate(out, 0)

        cell_emb = geneformer_embed(count_rows, symbols)
        embedding_source = "Geneformer V1-10M mean-pooled last hidden state"
        print(f"Geneformer embedding: {cell_emb.shape} | device: {device}")
        if device.type == "cuda":
            print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    except Exception as exc:
        print("Geneformer embedding 실패. fallback으로 전환합니다.")
        print(type(exc).__name__, exc)


다음은 fallback 경로입니다. `cell_emb`가 아직 `None`이면(Geneformer를 끄거나 로드가 실패한 경우) 앞에서 계산한 PCA 좌표를 임베딩으로 씁니다. 추려 둔 세포의 PCA representation을 가져와 같은 분석을 이어 갑니다. 이렇게 하면 모델 다운로드가 막혀도 시각화와 nearest neighbor 흐름이 끊기지 않습니다.

In [ ]:
if cell_emb is None:
    cell_emb = np.asarray(adata_hvg.obsm["X_pca"])[sub_idx]
    embedding_source = "fallback: PCA(30) baseline"
print("embedding source:", embedding_source, "| shape:", cell_emb.shape)


## 6. 세포 임베딩 시각화

임베딩을 2차원으로 투영해 그립니다. Geneformer 임베딩은 256차원이므로 PCA로 2차원으로 줄여 산점도를 그립니다. cell type 색으로 칠해 모델 임베딩이 면역세포 종류를 어떻게 배치하는지 봅니다. baseline UMAP과 비교하면 같은 세포를 두 표현이 어떻게 다르게 보는지 감을 잡을 수 있습니다.

In [ ]:
emb_xy = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(cell_emb)
plot_2d(emb_xy, sub_obs["cell_type"].to_numpy(), "cell embedding (2D projection) by cell type")


## 7. 비슷한 세포 찾기

임베딩 공간에서 각 세포의 가장 닮은 이웃을 찾습니다. `cosine_similarity`로 유사도를 구하고 대각선을 음의 무한대로 채워 자기 자신을 제외한 뒤, query와 nearest 세포의 cell type을 비교합니다.

**참고:** cosine similarity는 각도 기반으로 1이면 같은 방향, 0이면 직교, -1이면 반대입니다.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(cell_emb)
np.fill_diagonal(sim, -np.inf)
nn = sim.argmax(axis=1)
neighbor_df = pd.DataFrame({
    "query_type": sub_obs["cell_type"].to_numpy(),
    "nearest_type": sub_obs["cell_type"].to_numpy()[nn],
    "cosine_similarity": np.round(sim[np.arange(len(nn)), nn], 3),
})
agree = float((neighbor_df["query_type"] == neighbor_df["nearest_type"]).mean())
print(f"nearest neighbor가 같은 cell type인 비율: {agree:.2f}")
disagree = neighbor_df[neighbor_df["query_type"] != neighbor_df["nearest_type"]]
print(f"다른 cell type이 가장 가까운 세포: {len(disagree)}개 (아래는 그 예시 - 여기서 배울 게 많습니다)")
display(disagree.head(8) if len(disagree) else neighbor_df.head(8))


nearest neighbor가 같은 cell type인 비율이 높을수록 임베딩이 cell type 구조를 잘 보존합니다. 다만 이 가까움은 표현 안의 유사성이지 인과적 동일성이 아닙니다. 다른 type이 가까운 행은 marker 겹침, doublet, 중간 상태일 수 있습니다.

**참고:** 위 표는 일부러 불일치 세포만 보여 줍니다(예: T와 NK 혼동).

## 8. Supervised probing: 임베딩에 정보가 있는가

probing은 임베딩을 고정한 채 작은 분류기만 학습해 임베딩이 cell type 정보를 담는지 평가합니다. reference(train) 임베딩으로 logistic regression을 학습해 query(test) 정확도를 봅니다.

**주의:** 여기서는 가장 단순한 무작위 cell split을 씁니다. 슬라이드 p.40과 마무리 checklist의 donor/batch/tissue holdout이 실제로는 필요합니다. 또 pseudo-label과 임베딩이 같은 marker 발현에서 나와 검증이 부분적으로 순환적이므로, 점수는 정보 유무 정도로만 읽습니다.

In [ ]:
y = sub_obs["cell_type"].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(cell_emb, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
print("reference:", Xtr.shape[0], "| query:", Xte.shape[0])


reference 임베딩으로 logistic regression probe를 학습합니다. `class_weight="balanced"`는 cell type 불균형을 보정하고, `classification_report`로 type별 precision, recall, f1을 봅니다. 점수가 높아도 pseudo-label이 marker rule의 산물이라 생물학적 정확성을 증명하지는 않습니다.

**출력 읽기** support가 작은 type(예: Megakaryocyte)의 완벽 점수는 세포 하나만 틀려도 흔들리므로 과신하지 않습니다.

In [ ]:
probe = LogisticRegression(max_iter=2000, class_weight="balanced")
probe.fit(Xtr, ytr)
pred = probe.predict(Xte)
print(classification_report(yte, pred, zero_division=0))


혼동 행렬로 어떤 cell type을 어디로 헷갈렸는지 봅니다. 대각선이 맞게 분류한 경우이고, 대각선 밖 숫자가 혼동입니다. 면역세포 중 marker가 일부 겹치는 종류(예: T cell과 NK cell)가 서로 헷갈리는지, 드문 cell type이 흔한 type으로 흡수되는지를 보는 것이 점수 하나보다 유용합니다.

In [ ]:
labels_sorted = sorted(pd.unique(y))
cm = confusion_matrix(yte, pred, labels=labels_sorted)
display(pd.DataFrame(cm, index=[f"true {x}" for x in labels_sorted], columns=[f"pred {x}" for x in labels_sorted]))


## 9. Batch integration: 서로 다른 두 데이터셋

앞 절까지는 pbmc3k 한 데이터셋만 다뤘습니다. 여기서는 chemistry가 다른 pbmc_1k_v3(10x v3)를 pbmc3k(10x v1)와 함께 써서 실제 technical batch effect를 만듭니다.

- PCA baseline에서 두 batch가 갈라지는지.
- Geneformer zero-shot embedding이 두 batch를 섞는지.
- Harmony 같은 전용 도구와 비교.

**핵심:** Geneformer의 rank 인코딩은 depth 차이에 강건하지만 batch를 완전히 제거한다는 보장은 없으므로, 제거 정도를 지표로 확인합니다.

In [ ]:
# 두 번째 batch 데이터셋(pbmc_1k_v3, 10x v3) 다운로드 + QC + 주석
import os, urllib.request
import anndata as ad
_URL = "https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_filtered_feature_bc_matrix.h5"
_FN = "pbmc_1k_v3.h5"
def _annotate(a):
    pp = a.copy(); sc.pp.normalize_total(pp, target_sum=1e4); sc.pp.log1p(pp)
    sco = pd.DataFrame(index=a.obs_names)
    for ct, ms in MARKER_SETS.items():
        pr = [g for g in ms if g in pp.var_names]
        if pr:
            X = pp[:, pr].X
            sco[ct] = np.asarray(X.mean(axis=1)).ravel()
    a.obs["cell_type"] = sco.idxmax(axis=1).values
    return a
_brng = np.random.default_rng(RANDOM_STATE)   # 독립 rng (global rng 안 건드림 → 뒤 perturbation 결과 불변)
def _subs(a, n):
    idx = _brng.choice(a.n_obs, size=min(n, a.n_obs), replace=False)
    return a[idx].copy()

HAS_BATCH2 = False
try:
    if not os.path.exists(_FN):
        print("pbmc_1k_v3 다운로드 중 (~5MB)...")
        # 10x Cloudflare CDN은 기본 urllib UA를 403으로 막으므로 브라우저 UA 헤더 필요
        _req = urllib.request.Request(_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(_req, timeout=120) as _r, open(_FN, "wb") as _f:
            _f.write(_r.read())
    b2 = sc.read_10x_h5(_FN); b2.var_names_make_unique()
    sc.pp.filter_cells(b2, min_genes=200); sc.pp.filter_genes(b2, min_cells=3)
    b2.layers["counts"] = b2.X.copy(); _annotate(b2)

    b1 = adata.copy()                     # pbmc3k (raw counts in layers["counts"], gene_ids in var)
    if "cell_type" not in b1.obs: _annotate(b1)
    b1.obs["batch"] = "10x_v1"; b2.obs["batch"] = "10x_v3"
    N_PER = 350
    s1, s2 = _subs(b1, N_PER), _subs(b2, N_PER)
    # Geneformer는 세포별 독립 토큰화 → 배치가 유전자를 공유할 필요 없음. symbol 보존.
    s1.var["symbol"] = s1.var_names.astype(str); s2.var["symbol"] = s2.var_names.astype(str)
    gf1 = (s1.layers["counts"], np.asarray(s1.var["symbol"]))
    gf2 = (s2.layers["counts"], np.asarray(s2.var["symbol"]))
    # PCA/Harmony baseline은 Ensembl ID로 join (pbmc3k=hg19, v3=GRCh38 → symbol 말고 gene_ids)
    for a in (s1, s2):
        a.var_names = a.var["gene_ids"].astype(str).values; a.var_names_make_unique()
    comb = ad.concat([s1, s2], join="inner")
    comb.layers["counts"] = comb.X.copy()
    HAS_BATCH2 = True
    print("combined:", comb.shape, "| batch:", comb.obs["batch"].value_counts().to_dict())
    display(pd.crosstab(comb.obs["batch"], comb.obs["cell_type"]))
except Exception as e:
    print("batch2 로드 실패 — batch integration 데모를 건너뜁니다:", type(e).__name__, e)


먼저 baseline입니다. 두 데이터셋을 합쳐 표준 전처리(정규화·HVG·PCA)한 뒤, 같은 좌표를 batch 색과 cell type 색으로 봅니다. 2D 좌표는 UMAP으로 그리고, silhouette 지표는 전체 차원에서 계산합니다. batch가 갈라져 보이면 실제 batch effect가 있는 것입니다.

In [ ]:
# baseline: 합친 데이터에 표준 PCA. 2D는 UMAP, silhouette은 full-dim에서 계산.
from sklearn.metrics import silhouette_score as _ss
def _sil(X, key):
    return round(float(_ss(X, comb.obs[key])), 3)
def _project2d(X):
    try:
        import umap
        return umap.UMAP(n_components=2, random_state=RANDOM_STATE).fit_transform(X), "UMAP"
    except Exception:
        return PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X), "PCA(2)"
def show_method(emb, name):
    xy, proj = _project2d(emb)
    plot_2d(xy, comb.obs["batch"].to_numpy(), f"{name} - by BATCH ({proj})")
    plot_2d(xy, comb.obs["cell_type"].to_numpy(), f"{name} - by cell type ({proj})")
    print(f"[{name}] batch_sil={_sil(emb,'batch')} (낮을수록 섞임)  celltype_sil={_sil(emb,'cell_type')} (유지 좋음)")

results = {}
if HAS_BATCH2:
    cp = comb.copy(); sc.pp.normalize_total(cp, target_sum=1e4); sc.pp.log1p(cp)
    sc.pp.highly_variable_genes(cp, n_top_genes=2000)
    cp = cp[:, cp.var.highly_variable].copy(); sc.pp.scale(cp, max_value=10)
    X_pca = PCA(n_components=30, random_state=RANDOM_STATE).fit_transform(cp.X)
    results["PCA baseline"] = X_pca
    show_method(X_pca, "PCA baseline")


이제 Geneformer zero-shot 임베딩입니다. 두 batch를 각각 독립 토큰화·임베딩(같은 vocab)해 합친 뒤 batch 색과 cell type 색으로 봅니다. baseline보다 batch가 더 섞이면서 cell type이 유지되면 batch에 강건하다는 뜻입니다.

**참고:** fallback 경로에서는 실제 Geneformer가 없어 baseline과 같게 표시됩니다.

In [ ]:
# Geneformer 임베딩 (배치별 독립) → batch + cell type 색
if HAS_BATCH2:
    if str(embedding_source).startswith("Geneformer"):
        gf_emb = np.vstack([geneformer_embed(list(gf1[0].toarray() if hasattr(gf1[0], "toarray") else gf1[0]), gf1[1]),
                            geneformer_embed(list(gf2[0].toarray() if hasattr(gf2[0], "toarray") else gf2[0]), gf2[1])])
        _tag = "Geneformer (zero-shot)"
    else:
        gf_emb = results["PCA baseline"]   # fallback: 실제 Geneformer 없음 → baseline로 대체 표시
        _tag = "Geneformer (fallback=baseline)"
    results["Geneformer"] = gf_emb
    show_method(gf_emb, _tag)


전용 integration 도구 Harmony를 같은 데이터에 적용해, batch를 섞으면서 cell type을 보존하는지 봅니다.

In [ ]:
# Harmony (전용 integration 도구) — 없으면 설치, 실패해도 계속
if HAS_BATCH2:
    try:
        try:
            import harmonypy
        except ImportError:
            import subprocess, sys; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "harmonypy"]); import harmonypy
        _ho = harmonypy.run_harmony(results["PCA baseline"], comb.obs, ["batch"])
        _Z = np.asarray(_ho.Z_corr); results["Harmony"] = _Z if _Z.shape[0] == comb.n_obs else _Z.T
        show_method(results["Harmony"], "Harmony")
    except Exception as e:
        print("Harmony 건너뜀:", type(e).__name__, e)


각 방법을 한 표로 비교합니다. batch_silhouette은 낮을수록 좋고, celltype_silhouette은 baseline 수준을 유지해야 좋습니다. batch만 낮추고 celltype이 무너지면 over-integration입니다.

**핵심:** 좋은 integration은 batch_silhouette을 낮추면서 celltype_silhouette을 지킵니다.

In [ ]:
# 요약 표: 각 방법의 batch / cell type silhouette
if HAS_BATCH2 and results:
    tab = pd.DataFrame({k: {"batch_sil (↓좋음)": _sil(v, "batch"), "celltype_sil (유지좋음)": _sil(v, "cell_type")}
                        for k, v in results.items()}).T
    display(tab)


이 실습을 정리합니다.

- baseline에는 실제 v1/v3 batch가 있습니다(batch_sil > 0). 정규화가 depth 차이를 잡아 생각보다 크지 않을 수 있습니다.
- Geneformer zero-shot은 rank 인코딩 덕에 batch에 부분적으로 강건하지만 완전한 통합은 보장하지 못합니다.
- Harmony는 batch를 더 낮추면서 cell type을 보존해, 전용 도구가 왜 필요한지 보여 줍니다.

**핵심:** foundation model 임베딩도 zero-shot으로는 batch를 완전히 없애지 못하므로, 실제 데이터에서는 batch와 biology 지표를 함께 확인합니다. (batch_sil은 v1/v3의 cell type 구성 차이에도 영향받으니 celltype_sil과 같이 읽습니다.)

### Geneformer fine-tuning: 독립 데이터로 학습하면 통합·분류가 개선되나? (선택)

fine-tuning의 batch 통합 효과를 정직하게 보려면 **학습 데이터가 평가 batch에 들어가면 안 됩니다**(안 그러면 memorization). 그래서 역할을 나눕니다:

- **학습 = C (5k PBMC, 독립 셋)** — zero-shot probe(LR)와 fine-tuning 둘 다 C로 학습.
- **평가 = held-out A+B (v1+v3, §9와 동일한 두 batch)** — 통합(batch_sil·celltype_sil)과 분류(AUROC·accuracy)를 A·B는 한 번도 안 보고 측정.

**읽는 법:** fine-tuned의 batch_sil이 §9 Harmony급으로 내려가고 celltype_sil이 오르면 통합이 실제로 개선된 것. 분류는 **AUROC**로 봅니다 — accuracy는 class 불균형·decision에 민감해 AUROC와 함께 읽습니다. C가 v3라 A(v1)는 unseen chemistry이므로 **A/B를 나눠** 봅니다.

**주의:** label은 marker pseudo-label이라 부분적으로 순환적입니다. 실제 Geneformer(GPU)에서만 실행됩니다.

In [ ]:
# fine-tune 학습셋 C = 5k PBMC (독립). 평가셋 = held-out A+B (comb, v1+v3, §9와 동일).
RUN_FINETUNE = True
if RUN_FINETUNE and HAS_BATCH2 and str(embedding_source).startswith("Geneformer"):
    import os, urllib.request, time, random, torch, scipy.sparse as sp
    from transformers import AutoModelForSequenceClassification
    from sklearn.preprocessing import LabelEncoder
    _FT_SEED = 42   # fine-tuning 고정 seed
    random.seed(_FT_SEED); np.random.seed(_FT_SEED); torch.manual_seed(_FT_SEED); torch.cuda.manual_seed_all(_FT_SEED)
    _URLC, _FNC = "https://cf.10xgenomics.com/samples/cell-exp/3.0.2/5k_pbmc_v3/5k_pbmc_v3_filtered_feature_bc_matrix.h5", "5k_pbmc_v3.h5"
    if not os.path.exists(_FNC):
        print("C(5k) 다운로드 중 (~18MB)...")
        _rq = urllib.request.Request(_URLC, headers={"User-Agent": "Mozilla/5.0"})  # 10x CDN UA 우회
        with urllib.request.urlopen(_rq, timeout=180) as _r, open(_FNC, "wb") as _f: _f.write(_r.read())
    Cds = sc.read_10x_h5(_FNC); Cds.var_names_make_unique()
    sc.pp.filter_cells(Cds, min_genes=200); sc.pp.filter_genes(Cds, min_cells=3)
    Cds.layers["counts"] = Cds.X.copy(); _annotate(Cds)
    Cds = Cds[np.random.default_rng(_FT_SEED).choice(Cds.n_obs, min(800, Cds.n_obs), replace=False)].copy()
    print("C(train, 5k v3):", Cds.shape, "|", Cds.obs["cell_type"].value_counts().to_dict())

    _le = LabelEncoder().fit(np.array(list(MARKER_SETS.keys())))
    _PAD, _BS = token_dict.get("<pad>", 0), 16
    def _rw(X, i):
        r = X[i]; return np.asarray(r.todense()).ravel() if sp.issparse(r) else np.asarray(r).ravel()
    def _tk(counts, ens, ml=1024):
        nz = np.where(counts > 0)[0]
        pr = [(token_dict[ens[j]], counts[j]/median_dict[ens[j]]) for j in nz if ens[j] in token_dict and ens[j] in median_dict]
        pr.sort(key=lambda p: p[1], reverse=True); return [t for t, _ in pr[:ml]]
    def _btc(seqs, idxs, ys=None):
        ss = [seqs[i] or [_PAD] for i in idxs]; ml = max(len(x) for x in ss)
        ids = torch.tensor([x + [_PAD]*(ml-len(x)) for x in ss], device=device, dtype=torch.long)
        lab = torch.tensor([ys[i] for i in idxs], device=device) if ys is not None else None
        return ids, (ids != _PAD).long(), lab
    def _emb(model, seqs, is_cls):
        out = []
        with torch.no_grad():
            for _s in range(0, len(seqs), _BS):
                ids, att, _ = _btc(seqs, range(_s, min(_s+_BS, len(seqs))))
                h = (model(input_ids=ids, attention_mask=att, output_hidden_states=True).hidden_states[-1] if is_cls
                     else model(input_ids=ids, attention_mask=att).last_hidden_state)
                m = att.unsqueeze(-1).float(); out.append(((h*m).sum(1)/m.sum(1).clamp(min=1)).cpu().numpy())
        return np.concatenate(out, 0)
    _ensC = np.asarray(Cds.var["gene_ids"].astype(str)); _ensAB = np.asarray(comb.var_names)
    _tC = [_tk(_rw(Cds.layers["counts"], i), _ensC) for i in range(Cds.n_obs)]
    _tAB = [_tk(_rw(comb.layers["counts"], i), _ensAB) for i in range(comb.n_obs)]
    _yC = _le.transform(Cds.obs["cell_type"].to_numpy())
    print(f"토큰 커버리지  C / A+B : {np.mean([len(t)>0 for t in _tC]):.0%} / {np.mean([len(t)>0 for t in _tAB]):.0%}")

    _fm = AutoModelForSequenceClassification.from_pretrained(MODEL_REPO, subfolder=MODEL_SUBFOLDER, num_labels=len(_le.classes_)).to(device)
    _opt = torch.optim.AdamW(_fm.parameters(), lr=5e-5); _idx = np.arange(len(_tC)); _t0 = time.time()
    for _ep in range(3):
        _fm.train(); np.random.shuffle(_idx)
        for _s in range(0, len(_idx), _BS):
            ids, att, lab = _btc(_tC, _idx[_s:_s+_BS], _yC)
            o = _fm(input_ids=ids, attention_mask=att, labels=lab); o.loss.backward(); _opt.step(); _opt.zero_grad()
    _fm.eval(); print(f"[TIME] fine-tune on C(5k, {len(_tC)} cells): {time.time()-_t0:.1f}s")
else:
    print("fine-tuning은 실제 Geneformer(GPU) + HAS_BATCH2 경로에서만 실행됩니다.")


In [ ]:
# 평가: held-out A+B에서 통합(batch_sil·celltype_sil) + 분류(AUROC·accuracy, A/B 분해)
if RUN_FINETUNE and "_fm" in globals() and HAS_BATCH2 and str(embedding_source).startswith("Geneformer"):
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import label_binarize
    from sklearn.metrics import accuracy_score, roc_auc_score, silhouette_score
    def _proba(model, seqs):
        P = []
        with torch.no_grad():
            for _s in range(0, len(seqs), _BS):
                ids, att, _ = _btc(seqs, range(_s, min(_s+_BS, len(seqs))))
                P.append(torch.softmax(model(input_ids=ids, attention_mask=att).logits, -1).cpu().numpy())
        return np.concatenate(P, 0)
    def _auroc(y, P):
        cls = np.unique(y)
        if len(cls) < 2: return float("nan")
        try:
            Yb = label_binarize(y, classes=cls); Pp = P[:, cls]; Pp = Pp/Pp.sum(1, keepdims=True).clip(1e-9)
            return round(float(roc_auc_score(Yb, Pp, average="macro")), 3)
        except Exception: return float("nan")
    _sl = lambda e, l: round(float(silhouette_score(e, l)), 3) if len(np.unique(l)) >= 2 else float("nan")

    _zsAB, _ftAB = _emb(gf_model, _tAB, False), _emb(_fm, _tAB, True)
    _yAB = _le.transform(comb.obs["cell_type"].to_numpy()); _bAB = comb.obs["batch"].to_numpy()

    print("=== 통합 (held-out A+B)  batch_sil↓ 좋음 / celltype_sil↑ 좋음 ===")
    print(f"  Geneformer zero-shot : batch_sil={_sl(_zsAB,_bAB)}  celltype_sil={_sl(_zsAB,_yAB)}")
    print(f"  Geneformer fine-tuned: batch_sil={_sl(_ftAB,_bAB)}  celltype_sil={_sl(_ftAB,_yAB)}")
    if "Harmony" in results: print(f"  (§9 Harmony 참조)    : batch_sil={_sil(results['Harmony'],'batch')}  celltype_sil={_sil(results['Harmony'],'cell_type')}")

    _zsC = _emb(gf_model, _tC, False)
    _lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(_zsC, _yC)
    _pz, _pf = _lr.predict_proba(_zsAB), _proba(_fm, _tAB)
    _mA, _mB = _bAB == "10x_v1", _bAB == "10x_v3"
    print("\n=== 분류 (C로 학습 → held-out 평가)   accuracy / AUROC   (zero-shot probe → fine-tuned) ===")
    for _nm, _m in [("A+B", np.ones(len(_yAB), bool)), ("A (v1)", _mA), ("B (v3)", _mB)]:
        _y = _yAB[_m]
        _acc = f"{accuracy_score(_y, _pz[_m].argmax(1)):.3f}->{accuracy_score(_y, _pf[_m].argmax(1)):.3f}"
        _au = f"{_auroc(_y, _pz[_m])}->{_auroc(_y, _pf[_m])}"
        print(f"  {_nm:7s} accuracy {_acc:16s}  AUROC {_au}")
    _ft_int = _ftAB   # fine-tuned A+B 임베딩 (뒤 참고용)
else:
    print("이 셀은 fine-tune 셀 실행 후 실제 Geneformer(GPU) 경로에서만 동작합니다.")


### fine-tuning 후 in-silico perturbation은 어떻게 달라지나 (선택)

fine-tuned encoder로 cell-type program을 지우는 perturbation을 다시 계산해, 이동(effect_shift)이 zero-shot과 어떻게 다른지 봅니다.

- fine-tuning은 embedding을 cell-type 중심으로 재편하므로, cell-type program을 지우면 이동이 더 커집니다.
- 부분적으로 순환적입니다 — cell-type 라벨로 학습해 그 유전자에 민감해진 모델에서 같은 유전자를 지우기 때문입니다.

**핵심:** effect_shift의 절대값과 program 순위는 zero-shot이냐 fine-tuned냐에 따라 달라집니다. perturbation 결과는 model state에 의존하므로, 어떤 model인지 밝히지 않고 절대값을 비교하면 안 됩니다.

In [ ]:
# fine-tuning 후 perturbation effect_shift 비교 (선택) — fine-tune 셀 실행 후 실제 Geneformer(GPU)에서만
RUN_FT_PERTURB = True
if RUN_FT_PERTURB and "_fm" in globals() and "gf_model" in globals() and HAS_BATCH2 and str(embedding_source).startswith("Geneformer"):
    import torch, numpy as np, scipy.sparse as sp
    _ensP, _XcP = np.asarray(comb.var_names), comb.layers["counts"]
    _e2t = {e: (token_dict[e], median_dict[e]) for e in set(_ensP) if e in token_dict and e in median_dict}
    _PADp = token_dict.get("<pad>", 0)
    if "program_genes" not in globals():          # 섹션 10보다 먼저 실행돼도 안전 (동일 정의)
        _HKG = ["GAPDH", "ACTB", "B2M"]
        def program_genes(p): return MARKER_SETS.get(p, _HKG)
    def _rowP(i):
        r = _XcP[i]; return np.asarray(r.todense()).ravel() if sp.issparse(r) else np.asarray(r).ravel()
    def _tokP(c, drop=frozenset(), max_len=1024):
        nz = np.where(c > 0)[0]
        pr = [(_e2t[_ensP[j]][0], c[j]/_e2t[_ensP[j]][1]) for j in nz if _ensP[j] in _e2t and _ensP[j] not in drop]
        pr.sort(key=lambda p: p[1], reverse=True); return [t for t, _ in pr[:max_len]]
    def _embids(mdl, seqs, is_cls):
        out = []
        with torch.no_grad():
            for s in range(0, len(seqs), 16):
                ch = seqs[s:s+16]; ml = max(len(x) for x in ch)
                ids = torch.tensor([x + [_PADp]*(ml-len(x)) for x in ch], device=device, dtype=torch.long); att = (ids != _PADp).long()
                h = (mdl(input_ids=ids, attention_mask=att, output_hidden_states=True).hidden_states[-1] if is_cls
                     else mdl(input_ids=ids, attention_mask=att).last_hidden_state)
                m = att.unsqueeze(-1).float(); out.append(((h*m).sum(1)/m.sum(1).clamp(min=1)).cpu().numpy())
        return np.concatenate(out, 0)
    def _shift(mdl, is_cls, prog, n=30):
        ge = frozenset(name_id[g] for g in program_genes(prog) if g in name_id)
        cand = [i for i in range(comb.n_obs) if any(_ensP[j] in ge for j in np.where(_rowP(i) > 0)[0])]
        if len(cand) < 8: return None
        cells = list(np.random.default_rng(0).choice(cand, size=min(len(cand), n), replace=False))
        rows = [_rowP(i) for i in cells]
        base = _embids(mdl, [_tokP(r) for r in rows], is_cls)
        ko = _embids(mdl, [_tokP(r, drop=ge) for r in rows], is_cls)
        cos = (base*ko).sum(1) / (np.linalg.norm(base, axis=1)*np.linalg.norm(ko, axis=1) + 1e-8)
        return float(np.mean(1 - cos))
    print(f"{'program':14s} zero-shot  fine-tuned")
    for _p in MARKER_SETS:
        _z, _f = _shift(gf_model, False, _p), _shift(_fm, True, _p)
        if _z is not None and _f is not None:
            print(f"{_p:14s} {_z:.4f}     {_f:.4f}   {'↑' if _f > _z else ''}")
else:
    print("이 셀은 fine-tune 셀 실행 후 실제 Geneformer(GPU) 경로에서만 동작합니다.")


## 10. In silico perturbation

한 세포에서 어떤 gene(program)을 지우고 다시 임베딩을 뽑아, 원래에서 얼마나 멀어졌는지(1 − cosine = `effect_shift`)를 잽니다. 많이 움직이면 그 표현이 그 gene에 의존한다는 뜻이고, 거의 안 움직이면 덜 중요하다는 뜻입니다. 인과가 아니라 "그 program이 빠지면 표현이 얼마나 달라 보이나"입니다.

gene 하나만 빼면 세포당 수백 token 중 하나라 거의 안 움직이므로, 한 cell type을 정의하는 marker program 전체를 제거해 봅니다. Geneformer 경로면 토큰 제거 후 재임베딩, fallback이면 expression을 0으로 한 벡터 변화로 계산합니다.

**핵심:** 넓게 발현되는 program은 특이적이지 않아도 effect_shift가 클 수 있어 그 하나로는 부족하고, 한 cell type에 집중되는 정도인 specificity를 함께 봐야 합니다. 이 두 축이 이 섹션과 다음 우선순위화의 핵심입니다.

In [ ]:
GENEFORMER_OK = embedding_source.startswith("Geneformer")

# specificity는 정규화(log1p) 발현으로 계산 - annotation(섹션 4)과 같은 layer, library size 보정
def gene_mean_by_celltype(gene):
    if gene not in adata_pp.var_names:
        return None
    col = adata_pp[:, gene].X
    col = np.asarray(col.todense()).ravel() if sp.issparse(col) else np.asarray(col).ravel()
    return pd.Series(col).groupby(ct_arr).mean()

print("perturbation 경로:", "Geneformer 토큰 제거" if GENEFORMER_OK else "expression proxy")


`perturb_shift`는 program의 marker gene을 하나라도 발현하는 세포를 골라, 그 gene token을 모두 제거한 뒤 임베딩 이동(1 − cosine)을 평균으로 돌려줍니다. Geneformer 경로는 token을 빼고 재임베딩, fallback은 expression에서 그 gene을 0으로 만든 차이를 봅니다. 발현 세포가 없으면 결측입니다. 이 값은 인과가 아니라 그 program이 빠지면 표현이 얼마나 달라 보이는가입니다.

In [ ]:
def perturb_shift(genes, n=24):
    gjs = [int(np.where(symbols == g)[0][0]) for g in genes if len(np.where(symbols == g)[0])]
    if not gjs:
        return np.nan, 0
    cand = [i for i in sub_idx if any(dense_row(Mc, i)[gj] > 0 for gj in gjs)]
    if not cand:
        return np.nan, 0
    cells = list(rng.choice(cand, size=min(len(cand), n), replace=False))
    rows = [dense_row(Mc, i).astype(float) for i in cells]
    rows_ko = [r.copy() for r in rows]
    for r in rows_ko:
        for gj in gjs:
            r[gj] = 0.0
    if GENEFORMER_OK:
        base, pert = geneformer_embed(rows, symbols), geneformer_embed(rows_ko, symbols)
    else:
        def ev(r):
            s = r.sum(); return np.log1p(r / (s if s > 0 else 1) * 1e4)
        base = np.array([ev(r) for r in rows]); pert = np.array([ev(r) for r in rows_ko])
    cos = (base * pert).sum(1) / (np.linalg.norm(base, axis=1) * np.linalg.norm(pert, axis=1) + 1e-8)
    return float(np.mean(1.0 - cos)), len(cells)


교육용 program 후보 표를 만듭니다. 각 후보는 cell type program(marker gene set)과 literature support로 구성되고, 마지막 housekeeping(GAPDH·ACTB·B2M)은 negative control입니다. `program_genes`가 이름을 marker 목록으로 바꾸고 각 program에 `perturb_shift`로 effect를 계산합니다.

**주의:** `literature_support`(0.2~0.9)는 실습용 임의 예시이며 실제 evidence(Open Targets, GWAS 등)가 아닙니다.

In [ ]:
HK_GENES = ["GAPDH", "ACTB", "B2M"]
def program_genes(p):
    return MARKER_SETS.get(p, HK_GENES)

targets = pd.DataFrame([
    {"program": "T cell", "literature_support": 0.9},
    {"program": "B cell", "literature_support": 0.8},
    {"program": "Monocyte", "literature_support": 0.7},
    {"program": "NK cell", "literature_support": 0.6},
    {"program": "housekeeping", "literature_support": 0.2},
])
eff = [perturb_shift(program_genes(p)) for p in targets["program"]]
targets["effect_shift"] = [round(e[0], 4) if not np.isnan(e[0]) else np.nan for e in eff]
targets["n_cells"] = [e[1] for e in eff]
targets["genes"] = [", ".join(program_genes(p)) for p in targets["program"]]
display(targets[["program", "genes", "effect_shift", "n_cells", "literature_support"]])


### effect_shift를 embedding에서 시각화

각 program을 발현하는 세포가 유전자 제거 후 embedding에서 얼마나 이동하는지 화살표로 그립니다.

- 왼쪽(단일 gene 제거)은 화살표가 거의 없습니다.
- 오른쪽(program 전체 제거)은 화살표가 뚜렷하게 길어집니다.

**핵심:** 단일 gene 제거는 embedding을 거의 바꾸지 않고 program 제거는 크게 바꾸므로, program 단위로 지웁니다.

**주의:** 2D 화살표 길이는 256차원 이동의 투영이며 정확한 크기는 표의 cosine 값입니다.

In [ ]:
# in-silico perturbation을 임베딩 위에서 시각화: 모든 cell type program을 각각 (단일 gene vs program 전체)
def _embed_rows(rows):
    if str(embedding_source).startswith("Geneformer"):
        return geneformer_embed(rows, symbols)
    def ev(r):
        t = r.sum(); return np.log1p(r / (t if t > 0 else 1) * 1e4)
    return np.array([ev(r) for r in rows])
def _idx(g):
    hit = np.where(symbols == g)[0]
    return int(hit[0]) if len(hit) else None
def _ko(rows, genes):
    gj = [j for j in (_idx(g) for g in genes) if j is not None]
    out = [r.copy() for r in rows]
    for r in out:
        for j in gj:
            r[j] = 0.0
    return out
def _mcos(a, b):
    return float(np.mean(1 - (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-8)))

def viz_program(prog, n=40, min_cells=8):
    genes = program_genes(prog); single = genes[0]
    gj_all = [j for j in (_idx(g) for g in genes) if j is not None]
    cand = [i for i in sub_idx if any(dense_row(Mc, i)[j] > 0 for j in gj_all)]
    if len(cand) < min_cells:
        print(f"[{prog}] 발현 세포 {len(cand)}개 — 너무 적어 생략"); return
    cells = list(rng.choice(cand, size=min(len(cand), n), replace=False))
    rows = [dense_row(Mc, i).astype(float) for i in cells]
    base = _embed_rows(rows); ko_s = _embed_rows(_ko(rows, [single])); ko_p = _embed_rows(_ko(rows, genes))
    pc = PCA(n_components=2, random_state=RANDOM_STATE).fit(base)
    b2, s2, p2 = pc.transform(base), pc.transform(ko_s), pc.transform(ko_p)
    ms, mp = _mcos(base, ko_s), _mcos(base, ko_p)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)
    for ax, dst, ttl in [(axes[0], s2, f"single-gene KO: {single}  (shift={ms:.4f})"),
                         (axes[1], p2, f"program KO: {prog}  (shift={mp:.4f})")]:
        ax.scatter(b2[:, 0], b2[:, 1], s=20, c="lightgray", zorder=1)
        ax.quiver(b2[:, 0], b2[:, 1], dst[:, 0] - b2[:, 0], dst[:, 1] - b2[:, 1],
                  angles="xy", scale_units="xy", scale=1, color="crimson", width=0.005, alpha=0.85, zorder=2)
        ax.set_title(ttl, fontsize=10); ax.set_xlabel("embedding axis 1")
    axes[0].set_ylabel("embedding axis 2")
    fig.suptitle(f"[{prog}]  arrows = cell displacement | left: single gene (tiny) -> right: whole program (large)  (n={len(cells)})", fontsize=11)
    plt.tight_layout(); plt.show()
    print(f"  [{prog}] shift: single '{single}'={ms:.4f}  vs  program={mp:.4f}  (배율 {mp/ms:.1f}x)" if ms > 0 else f"  [{prog}] single={ms:.4f} program={mp:.4f}")

# 모든 cell type program을 순회 (housekeeping 음성대조 제외)
for _p in MARKER_SETS:
    viz_program(_p)


effect_shift가 큰 program일수록 marker set 제거가 표현을 크게 흔듭니다. 함정은, effect_shift만으로 순위를 매기면 특이적이지 않아도 넓게 발현되는 program이 위로 올라올 수 있다는 점입니다.

**주의:** Geneformer는 raw 발현량이 아니라 `count / gene median` 순위를 쓰므로, GAPDH나 B2M 같은 housekeeping은 median이 커서 오히려 순위가 낮아집니다(섹션 3의 정규화 효과). housekeeping이라 rank 최상위라는 직관은 맞지 않습니다.

핵심은 **rank(순위)와 deletion 효과(effect_shift)가 별개의 축**이라는 점입니다. 순위가 낮은 gene도 지웠을 때 표현이 흔들릴 수 있고 그 반대도 가능하므로, effect_shift 하나로는 부족하고 specificity(한 cell type에 집중되는 정도)를 함께 봐야 cell-type-specific program이 드러납니다.

**핵심:** effect_shift가 크다고 치료 표적은 아닙니다. 우선순위화의 단서일 뿐, 실제 인과는 held-out perturbation과 functional assay가 필요합니다.

## 11. Target prioritization

effect_shift에 specificity를 더해 우선순위 점수를 만듭니다. specificity는 marker 발현이 한 cell type에 얼마나 집중되는지로, cell type별 평균 발현의 최대값을 합으로 나눈 값입니다. 1에 가까울수록 특이적이고(cell-type program), 고르게 퍼지면 낮습니다(housekeeping).

**주의:** 여기서 specificity는 통계의 특이도 TN/(TN+FP)가 아니라 발현이 한 cell type에 집중된 정도입니다.

In [ ]:
def specificity(program):
    accum = None
    for g in program_genes(program):
        s = gene_mean_by_celltype(g)
        if s is None:
            continue
        accum = s.copy() if accum is None else accum.add(s, fill_value=0)
    if accum is None or float(accum.sum()) == 0:
        return 0.0
    return float(accum.max() / accum.sum())

targets["specificity"] = [round(specificity(p), 3) for p in targets["program"]]
display(targets[["program", "effect_shift", "specificity", "literature_support"]])


effect_shift와 specificity를 0~1로 정규화하고 literature support를 더한 가중합으로 priority score를 내며, effect 미측정이나 specificity가 낮으면 failure flag를 답니다. 점수만 보지 말고 플래그를 함께 읽습니다.

**주의:** 가중치(effect 1.0, specificity 0.8, literature 0.5, flag -0.5)는 실습용 예시입니다. effect·specificity는 이 5개 후보 안에서 정규화하므로 점수는 상대값이라 절대 점수로 읽으면 안 됩니다.

In [ ]:
def norm01(s):
    s = pd.Series(s, dtype=float)
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else pd.Series(0.0, index=s.index)

rank_df = targets.copy()
rank_df["eff_norm"] = norm01(rank_df["effect_shift"].fillna(0))
rank_df["spec_norm"] = norm01(rank_df["specificity"])
rank_df["flags"] = [
    "; ".join(
        ([] if r["n_cells"] > 0 and not np.isnan(r["effect_shift"]) else ["not_detected"])
        + ([] if r["specificity"] >= 0.3 else ["low_specificity"])
    ) or "(none)"
    for _, r in rank_df.iterrows()
]
rank_df["n_flags"] = rank_df["flags"].apply(lambda s: 0 if s == "(none)" else s.count(";") + 1)


In [ ]:
rank_df["priority_score"] = (
    rank_df["eff_norm"] + 0.8 * rank_df["spec_norm"] + 0.5 * rank_df["literature_support"]
    - 0.5 * rank_df["n_flags"]
)
rank_df = rank_df.sort_values("priority_score", ascending=False).reset_index(drop=True)
rank_df["rank"] = np.arange(1, len(rank_df) + 1)
display(rank_df[["rank", "program", "effect_shift", "specificity", "literature_support", "flags", "priority_score"]].round(3))


이 표가 핵심 산출물입니다. priority_score 숫자만 보지 말고 단서와 failure flag를 함께 봅니다. effect_shift만 보면 발현량이 큰 housekeeping program이 올라오지만 specificity가 낮아(low_specificity 플래그) 최종 priority에서는 내려가므로, effect와 specificity를 함께 써야 cell-type-specific program이 상위로 옵니다.

**주의:** 이 순위는 follow-up 실험 우선순위이지 치료 표적이 아닙니다. therapeutic claim으로 바꾸려면 perturb-seq, functional assay, held-out validation이 필요합니다.

## 12. single-cell foundation model 개요

오늘 Geneformer를 직접 다뤘지만, single-cell foundation model은 입력 방식에 따라 여러 갈래가 있습니다. 모델 카드를 읽을 때 cell을 어떻게 token으로 바꾸는지 먼저 보면 결과 해석의 기준이 잡힙니다. 아래 표로 대표 모델을 정리합니다.

In [ ]:
model_overview = pd.DataFrame([
    {"모델": "scBERT", "입력": "binned expression", "특징": "초기 scFM, efficient attention"},
    {"모델": "Geneformer", "입력": "rank-ordered genes", "특징": "약 30M cells, in silico perturbation"},
    {"모델": "scGPT", "입력": "gene + expression + condition", "특징": "generative, annotation·integration·perturbation"},
    {"모델": "scFoundation / CellFM", "입력": "large-scale corpus", "특징": "scale-up, multi-task transfer"},
])
display(model_overview)


## 13. 정리: 무엇을 만들었고 무엇을 말하면 안 되는가

PBMC3k를 전처리하고 rank-value tokenization을 확인한 뒤 실제 Geneformer로 임베딩을 추출하고, 그 위에서 marker annotation, supervised probing, batch integration, in silico perturbation, target prioritization을 수행했습니다. 점수는 후보를 정렬하거나 표현을 비교하는 도구였지 인과나 임상 결론의 증거가 아니었습니다.

**핵심:** 이 노트북의 점수는 후보를 정렬하고 표현을 비교하는 도구이며 인과의 증거가 아닙니다.

### scFM 결과를 읽는 순서

1. 입력. counts/normalized, symbol/Ensembl ID, metadata 품질.
2. tokenization. rank인지 gene+value인지, 무엇을 보존하고 버리는지.
3. embedding. baseline(PCA/HVG)과 비교해 무엇을 더하는지.
4. annotation/integration. reference coverage, confidence, batch mixing, cell-state 보존.
5. 검증. split(donor/batch/tissue holdout), baseline(scVI/Harmony), claim level.

### perturbation 결과로 말할 수 있는 것과 아직 말하면 안 되는 것

같은 embedding shift라도 허용되는 주장 수준은 다릅니다.

- embedding shift는 이 gene을 제거하면 해당 cell type 표현에서 멀어진다는 모델 출력 사실입니다.
- state hypothesis는 그 cell state 신호가 약해진다는 후보 가설이며 검증이 필요합니다.
- network hypothesis는 pathway gene이 함께 변한다는 후보이며, co-expression은 regulation이 아닙니다.
- therapeutic claim은 독립 perturb-seq와 functional assay 없이는 말하지 않습니다.

### 자주 생기는 해석 오류

- UMAP/cluster 분리를 cell type 정답으로 단정.
- annotation accuracy가 높으면 marker pseudo-label이 생물학적으로 옳다고 결론.
- batch silhouette만 보고 integration 성공이라 판단(cell-state 보존을 놓침).
- rare cell type의 불안정한 embedding·confidence를 무시.
- perturbation embedding shift를 곧장 therapeutic target으로 연결.

### 확인 문제

1. cell × gene matrix에 자연스러운 순서가 없다는 것이 tokenizer 설계에 어떤 자유와 가정을 만드는가?
2. Geneformer rank-ordering과 scGPT gene+expression은 각각 무엇을 보존하고 버리는가?
3. annotation accuracy가 높아도 임베딩을 batch 색으로 다시 봐야 하는 이유는?
4. batch integration에서 batch silhouette만 낮추면 안 되는 이유는?
5. embedding shift를 therapeutic target claim으로 말하면 안 되는 이유를 claim level로 설명하라.
6. 실제 disease 데이터로 확장할 때 gene vocabulary·reference coverage·batch·donor 중 무엇을 먼저 점검하겠는가?